# CS328 Introduction to Data Science - Homework 1
by: Karan Gandhi - 23110157


## Problem 1

Suppose you define a clustering objective in the following manner - give a partition $\mathcal{C} = \{C_1, \dots, C_k\}$, define 

$$cost(\mathcal{C}) = \sum_i \frac{1}{|C_i|} \sum_{x, y \in C_i} ||x - y||^2_2$$

i.e. cost of a cluster is the sum of all pairwise squared distances. Give an algorithm for this.

### Solution:

Let us first simplify the cost function. We can rewrite the cost function as follows:

$$
\begin{align*}
cost(\mathcal{C}) &= \sum_i \frac{1}{|C_i|} \sum_{x, y \in C_i} ||x - y||^2_2 \\
&= \sum_i \frac{1}{|C_i|} \sum_{x \in C_i} \sum_{y \in C_i} ||x - \mu_i + \mu_i - y||^2_2 \\
&= \sum_i \frac{1}{|C_i|} \sum_{x \in C_i} \sum_{y \in C_i} ||x - \mu_i||^2_2 + ||\mu_i - y||^2_2 + 2(x - \mu_i)^T(\mu_i - y) \\
&= \sum_i \frac{1}{|C_i|} (\sum_{x \in C_i} |C_i|\cdot||x - \mu_i||^2_2) + (\sum_{y \in C_i} |C_i|\cdot||\mu_i - y||^2_2) + \sum_{x \in C_i} \sum_{y \in C_i} 2(x - \mu_i)^T(\mu_i - y) \\
&= \sum_i \sum_{x \in C_i} ||x - \mu_i||^2_2 + \sum_{y \in C_i} ||\mu_i - y||^2_2 + \frac{1}{|C_i|} \sum_{x \in C_i} \sum_{y \in C_i} 2(x - \mu_i)^T(\mu_i - y) \\
&= \sum_i 2 \cdot \sum_{x \in C_i} ||x - \mu_i||^2_2 + \frac{1}{|C_i|} \sum_{x \in C_i} \sum_{y \in C_i} 2(x - \mu_i)^T(\mu_i - y) \\
&= \sum_i 2 \cdot \sum_{x \in C_i} ||x - \mu_i||^2_2 + \frac{1}{|C_i|} \sum_{x \in C_i} 2(x - \mu_i)^T\sum_{y \in C_i} (\mu_i - y) \\
&= \sum_i 2 \cdot \sum_{x \in C_i} ||x - \mu_i||^2_2 + 0 \\
&= 2 \cdot \sum_i \sum_{x \in C_i} ||x - \mu_i||^2_2 \\
&= 2 \cdot cost_{kmeans}(\mathcal{C})
\end{align*}$$

Here $\mu_i$ is the centroid of cluster $C_i$.

As you can see, the given cost function is just twice the cost function for k-means. So since the objective function only differs by a constant factor, we can use Loyd's algorithm or kmeans++ to solve this problem.

Here is the pseudocode for Lloyd's algorithm is as follows:

1. Initialize centroids $\mu_1, \mu_2, \dots, \mu_k$ randomly.
2. Repeat until convergence:
	a. For each data point $x_i$, assign it to the nearest centroid:
		$$C_j = \{ x_i : ||x_i - \mu_j||^2 \leq ||x_i - \mu_l||^2 \text{ for all } l, 1 \leq l \leq k \}$$
	b. Update the centroids:
		$$\mu_j = \frac{1}{|C_j|} \sum_{x_i \in C_j} x_i$$

## Problem 2
For the k-means problem, show that there is at most a factor of four ratio between the optimal value when we either require all cluster centers to be data points or allow arbitrary points to be centers.

### Solution

Let $OPT_{restrict} = \{C_1', \dots, C_k'\}$ be the optimal value of the k-means problem when we require all cluster centers to be data points and $OPT = \{C_1^*, \dots, C_k^*\}$ be the optimal value when we allow arbitrary points to be centers. We have to prove the following inequality:

$$
\begin{align*}
    &\frac{cost(OPT_{restrict})}{cost(OPT)} \le 4\\
    \implies& cost(OPT_{restrict}) \le 4 \cdot cost(OPT)
\end{align*}
$$

Where $cost(S)$ is the cost of the solution $S$ defined as:

$$cost(S) = \sum_i \sum_{x \in P_i} ||x - C_i||^2_2$$

Where $P_i$ is the set of points that have $C_i$ as the closest centriod in the solution $S$.

Let us make another solution $S' = \{C_1, \dots, C_k\}$ where $C_i$ is the point that is closest to $C_i^*$. Since we are still restricting the centroids to only the data points, the cost of this solution will be atleast the optimal cost when we restrict the centroids only to the data points. Hence the following inequality holds:

$$
\begin{align*}
    cost(OPT_{restrict}) \le cost(S')
\end{align*}
$$

Let $P_i$ be the set of points that have $C_i^*$ as the closest centriod, and $P'_i$ be the set of points that have $C_i$ as the closist centroid. Now, we know that the following inequality for $cost(S)$ holds:

$$
\begin{align*}
    cost(S') &\le \sum_i \sum_{x \in P'_i} ||x - C_i||^2_2 \le \sum_i \sum_{x \in P_i} ||x - C_i||^2_2\\
\end{align*}
$$

This is because if the point $x$ is not assigned to the closest centroid $C_i$, then the cost will increase ($\forall x \in P'_i, ||x - C_i||_2 \le ||x - C_j||_2$ for some $j$). Now, we can write the above equation as:

$$
\begin{align*}
    cost(S') &\le \sum_i \sum_{x \in P_i} ||x - C_i^* + C_i^* - C_i||^2_2\\
    &\le \sum_i \sum_{x \in P_i} (||x - C_i^*||^2_2 + ||C_i^* - C_i||^2_2 + 2(x - C_i^*)^T(C_i^* - C_i))\\
    &\le \sum_i \sum_{x \in P_i} (||x - C_i^*||^2_2 + ||C_i^* - C_i||^2_2 + 2\cdot||x - C_i^*||_2\cdot||C_i^* - C_i||_2)\\
    &\le \sum_i \sum_{x \in P_i} (||x - C_i^*||_2 + ||C_i^* - C_i||_2)^2\\
\end{align*}
$$

So we have: 
$$
\begin{align}
    cost(S') \le \sum_i \sum_{x \in P_i} (||x - C_i^*||_2 + ||C_i^* - C_i||_2)^2
\end{align}
$$

Now for any point x which has $C_i^*$ as the closest centroid, we have:

$$
||C_i - C_i^*||_2 \le ||x - C_i^*||_2
$$
This is true because $C_i$ is the point closest to $C_i^*$, so all points that have $C_i^*$ as the closest centroid will have a distance atleast $||C_i - C_i^*||_2$ from $C_i^*$.
Substituting this in equation (1), we get:

$$
\begin{align*}
    cost(S') &\le \sum_i \sum_{x \in P_i} (||x - C_i^*||_2 + ||C_i^* - C_i||_2)^2\\
    &\le \sum_i \sum_{x \in P_i} (||x - C_i^*||_2 + ||C_i^* - x||_2)^2\\
    &\le \sum_i \sum_{x \in P_i} (2\cdot||x - C_i^*||_2)^2\\
    &\le 4 \cdot \sum_i \sum_{x \in P_i} ||x - C_i||_2^2\\
    &\le 4 \cdot cost(OPT)
\end{align*}
$$

So we have:

$$
	cost(OPT_{restrict}) \le cost(S') \le 4 \cdot cost(OPT)
$$

Hence Proved.




## Problem 3

Create a random variable X for which Markov’s inequality is tight. Give proof for your answer. If it is tight, then why are we using other inequalities e.g. Chebyshev and Chernoff?

### Solution

For the sake of simplicity, let's take a random variable $X$ which only takes 2 positive values. Let $X$ take values $0$ and $1$ with probabilities $1 - p$ and $p$ respectively. More formally:

$$
X = 
\begin{cases} 
0 & \text{with probability } 1 - p \\
1 & \text{with probability } p 
\end{cases}
$$

We want the Markov's inequality to be tight, so we want the following equality to hold for some $a$:

$$
\begin{align*}
	&P(X \ge a) = \frac{E(X)}{a}\\
	\implies & P(X \ge a) = \frac{p}{a}
\end{align*}$$
Choosing $a = 1$, we get the folllowing equation:

$$
\begin{align*}
	&P(X \ge 1) = \frac{p}{1}\\
\end{align*}
$$

But $P(X \ge 1) = p$, so the equation simplifies to:

$$
\begin{align*}
	p = \frac{p}{1}
\end{align*}
$$

Hence the Markov's inequality is tight for this random variable $X$, for all $p \in [0, 1]$ at $a = 1$. However, this is only specific to this random variable $X$ and not all random variables. Moreover, this tightness is only at a specific $a$ ($a = 1$). If we try to build a random variable for which Markov's inequality is tight for all $a$, we will not be able to do so. Here is the proof for the same:

Let's take a continious random variable $X$ which only takes positive values, having a probability density function $p_X(x) = f(x)$. Since we want the Markov's inequality to be tight, we want the following equality to hold:

$$
\begin{align*}
	&P(X \ge a) = \frac{E[X]}{a} \\
    \implies& \int_a^{\infty} f(x)dx = \frac{\int_{0}^{\infty} x\cdot f(x)dx}{a}
\end{align*}
$$

Differentiating both sides with respect to a, we get:

$$-f(a) = -\frac{\int_{0}^{\infty} x\cdot f(x)dx}{a^2}$$
$$f(a) = \frac{\int_{0}^{\infty} x\cdot f(x)dx}{a^2}$$

So our function $f(x)$ must be of the following form:

$$f(x) = \frac{c}{x^2}$$

Where c is a constant and is equal to $E(x)$. Now, we know that the probability density function must integrate to 1, so we have:

$$
\begin{align*}
    &\int_{0}^{\infty} f(x)dx = 1\\
    \implies& \int_{0}^{\infty} \frac{c}{x^2}dx = 1\\
\end{align*}
$$

However, the above integral is divergent, so the probability density function $f(x)$ cannot exist.

Hence there is no continious random variable $X$ which only takes positive values, such that the Markov's inequality is tight for all values of $a$.

Now to show the weakness of Markov's inequality, let's take a exponential random variable $X$ with parameter $\lambda$ as 1. The probability density function of $X$ is given by:

$$
f(x) = 
\begin{cases} 
e^{-x} & \text{if } x \ge 0 \\
0 & \text{otherwise}
\end{cases}
$$


We know that the expected value of $X$ is $\lambda$ which is 1 in this case.

Now applying the markov inequality at $a = 100$ we get the following:

$$
\begin{align*}
	&P(X \ge 100) \le \frac{E[X]}{100}\\
	\implies& P(X \ge 100) \le \frac{1}{100}
\end{align*}
$$

Now, the probability that $X$ is greater than 100 is given by:

$$
\begin{align*}
	&P(X \ge 100) = \int_{100}^{\infty} e^{-x}dx\\
	\implies& P(X \ge 100) = e^{-100}
\end{align*}
$$

Now as you can see $e^{-100} << \frac{1}{100}$, So we can say that the Markov's gave a very weak bound in this case. This is why we use other inequalities like Chebyshev and Chernoff which give better bounds.

If we were to use Chebyshev's inequality, we would get the following:

$$
\begin{align*}
	&P(|X - E[X]| \ge a) \le \frac{Var(X)}{a^2}\\
	\implies& P(|X - 1| \ge a) \le \frac{1}{a^2}
\end{align*}
$$

putting $a = 100$, we get:

$$
\begin{align*}
	&P(|X - 1| \ge 100) \le \frac{1}{100^2}\\
	\implies& P(|X - 1| \ge 100) \le \frac{1}{10000}
\end{align*}
$$

which is a much better bound than the Markov's inequality. This is because Chebyshev using the information from the second moment of X. Chernoff gives a better tailing bound for the variable. So in general, we use other inequalities like Chebyshev and Chernoff are used because they give better bounds than Markov's inequality.

## Problem 4

Download the MNIST dataset from http://yann.lecun.com/exdb/mnist/. We will use the test dataset and test labels only.

(a) Cluster them first using k-means clustering, k = 10, with kmeans++ initialization (implement the complete Lloyd’s algorithm yourself). Check the Rand-index of the clustering against the true labels. Use the sklearn module for rand-index.

(b) Do the same for k-center clustering, k = 10. Implement the greedy algorithm discussed in class. Report the Rand-index here too.

(c) Run the single linkage agglomeration till there are k = 10 clusters. Report Rand-index here too.

(d) Run the same algorithms (k-means and k-center) but on a rank-k approximation of the training data matrix. Note that if A is the training data matrix (images × pixels), then you can just use $U_k, \Sigma_k$ for the clustering, no need to use $V_k$. Evaluate for k = 2, 5, 10 and report the rand-index values.

### Imports

In [55]:
import numpy as np
import struct
from array import array
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.cluster import rand_score, adjusted_rand_score
import pandas as pd
from keras.datasets import mnist

### Reading the data

Note we will download the data from: https://www.kaggle.com/datasets/hojjatk/mnist-dataset, and put it in the data folder. If you don't want to do this then run the next cell.

In [38]:
# Taken and modified from https://www.kaggle.com/code/hojjatk/read-mnist-dataset

class MnistDataloader(object):
    def __init__(self, test_images_filepath, test_labels_filepath):
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath
    
    def read_images_labels(self, images_filepath, labels_filepath):        
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            labels = array("B", file.read())        
        
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            image_data = array("B", file.read())        
        images = []
        for i in range(size):
            images.append([0] * rows * cols)
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img            
        
        return images, labels
            
    def load_data(self):
        x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return np.array(x_test), np.array(y_test)


In [43]:
test_images_filepath = './data/t10k-images-idx3-ubyte/t10k-images-idx3-ubyte'
test_labels_filepath = './data/t10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte'

dataloader = MnistDataloader(test_images_filepath, test_labels_filepath)
x_test, y_test = dataloader.load_data()
x_test = x_test.reshape(x_test.shape[0], -1)



Note if you are running this on colab run the following cell to load the data

In [56]:
_, (x_test, y_test) = mnist.load_data()
x_test = x_test.reshape(x_test.shape[0], -1)
x_test = x_test.astype('float32') / 255

In [44]:
def print_metrics(y_true, y_pred):
	print('Rand Index:', rand_score(y_true, y_pred))
	print('Adjusted Rand Index:', adjusted_rand_score(y_true, y_pred))

### K-Means plus plus

In [45]:
def k_means_pp_clustering(X, k, max_iter=20):
	n = X.shape[0]
	centroids = np.zeros((k, X.shape[1]))
	centroids[0] = X[np.random.choice(n)]
	for i in range(1, k):
		distances = np.zeros((n, i))
		for j in range(i):
			distances[:, j] = np.linalg.norm(X - centroids[j], axis=-1)
		probs = np.min(distances, axis=-1) ** 2
		probs /= probs.sum()
		centroids[i] = X[np.random.choice(n, p=probs)]

	for _ in range(max_iter):
		distances = np.zeros((n, k))
		for i in range(k):
			distances[:, i] = np.linalg.norm(X - centroids[i], axis=-1)
		labels = np.argmin(distances, axis=-1)
		new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
		if np.mean(centroids == new_centroids) == 1:
			break # break if all centroids are the same
		centroids = new_centroids
	return labels, centroids

In [46]:
labels, centriods = k_means_pp_clustering(x_test, 10)
print("Metrics for KMeans++")
print_metrics(y_test, labels)

Metrics for KMeans++
Rand Index: 0.8881849584958496
Adjusted Rand Index: 0.41007375682828806


### K Center Clustering

In [47]:
def k_center_clustering(X, k):
	centroids = np.zeros((k, X.shape[1]))
	centroids[0] = X[np.random.choice(X.shape[0])]
	
	for i in range(1, k):
		distances = np.zeros(X.shape[0])
		for j, point in enumerate(X):
			distances[j] = np.min(np.linalg.norm(point - centroids[:i], axis=-1))
		centroids[i] = X[np.argmax(distances)]
	labels = np.zeros(X.shape[0])
	for i, point in enumerate(X):
		labels[i] = np.argmin(np.linalg.norm(point - centroids, axis=-1))
	return labels, centroids

In [73]:
labels, centriods = k_center_clustering(x_test, 10)
print("Metrics for KCenter")
print_metrics(y_test, labels)

Metrics for KCenter
Rand Index: 0.7532670667066707
Adjusted Rand Index: 0.11098353037487248


### Single link aggolomerative clustering

In [49]:
def single_link_aggolomeration(X, k):
	n = X.shape[0]
	distances = np.zeros((n, n))
	for i in range(X.shape[0]):
		distances[i] = np.linalg.norm(X - X[i], axis=-1)

	clusters = [[x] for x in range(n)]
 
	while (len(clusters) != k):
		min_distance = np.inf
		cluster1 = -1
		cluster2 = -1
		for i in range(len(clusters)):
			for j in range(i+1, len(clusters)):
				for x in clusters[i]:
					for y in clusters[j]:
						if distances[x, y] < min_distance:
							min_distance = distances[x, y]
							cluster1 = i
							cluster2 = j
		clusters[cluster1].extend(clusters[cluster2])
		clusters.pop(cluster2)
  
	labels = np.zeros(n)
	for i, cluster in enumerate(clusters):
		for point in cluster:
			labels[point] = i

	return labels, clusters

In [50]:
labels, clusters = single_link_aggolomeration(x_test[:1000], 10)
print("Metrics for Single Link Agglomeration")
print_metrics(y_test[:1000], labels)

Metrics for Single Link Agglomeration
Rand Index: 0.11554354354354354
Adjusted Rand Index: 0.00037373131203754167


### Single link aggolomerative clustering using sklearn

In [51]:
agg_clustering = AgglomerativeClustering(n_clusters=10, linkage='single')
labels = agg_clustering.fit_predict(x_test[:1000])

print("Metrics for Single Link Agglomeration using sklearn")
print_metrics(y_test[:1000], labels)

Metrics for Single Link Agglomeration using sklearn
Rand Index: 0.11574774774774775
Adjusted Rand Index: 0.0006045260393629259


### K-Means and K-center on a Rank-k approximation of the data

In [82]:
def get_rank_k_approx(x, k):
	u, s, vh = np.linalg.svd(x, full_matrices=False)
	u_rank = u[:, :k]
	sigma_rank = np.diag(s[:k])
	
	return u_rank @ sigma_rank

In [86]:
results = []

k = [2, 5, 10]

for i in k:
	print(f'Rank k approximation for k = {i}')
	x_test_approx = get_rank_k_approx(x_test, i)
	
	labels, centriods = k_means_pp_clustering(x_test_approx, 10)
	rand_index = rand_score(y_test, labels)
	adjusted_rand_index = adjusted_rand_score(y_test, labels)
	results.append({'Method': 'kmeans++', 'k': i, 'Rand Index': rand_index, 'Adjusted Rand Index': adjusted_rand_index})
	print(f'Rand Index for kmeans++ with rank k approximation: {rand_index}')
	print(f'Adjusted Rand Index for kmeans++ with rank k approximation: {adjusted_rand_index}')
	
	labels, centriods = k_center_clustering(x_test_approx, 10)
	rand_index = rand_score(y_test, labels)
	adjusted_rand_index = adjusted_rand_score(y_test, labels)
	results.append({'Method': 'kcenter', 'k': i, 'Rand Index': rand_index, 'Adjusted Rand Index': adjusted_rand_index})
	print(f'Rand Index for kcenter with rank k approximation: {rand_index}')
	print(f'Adjusted Rand Index for kcenter with rank k approximation: {adjusted_rand_index}')

df_results = pd.DataFrame(results)
display(df_results)

Rank k approximation for k = 2
Rand Index for kmeans++ with rank k approximation: 0.8308436843684368
Adjusted Rand Index for kmeans++ with rank k approximation: 0.1410568544944035
Rand Index for kcenter with rank k approximation: 0.7311523552355236
Adjusted Rand Index for kcenter with rank k approximation: 0.09284138062075858
Rank k approximation for k = 5
Rand Index for kmeans++ with rank k approximation: 0.8707854785478548
Adjusted Rand Index for kmeans++ with rank k approximation: 0.3056086909580594
Rand Index for kcenter with rank k approximation: 0.7461872587258725
Adjusted Rand Index for kcenter with rank k approximation: 0.10530202795700905
Rank k approximation for k = 10
Rand Index for kmeans++ with rank k approximation: 0.8766212421242124
Adjusted Rand Index for kmeans++ with rank k approximation: 0.3293314050947401
Rand Index for kcenter with rank k approximation: 0.799211901190119
Adjusted Rand Index for kcenter with rank k approximation: 0.18088792948341104


,Method,k,Rand Index,Adjusted Rand Index
0,kmeans++,2,0.830844,0.141057
1,kcenter,2,0.731152,0.092841
2,kmeans++,5,0.870785,0.305609
3,kcenter,5,0.746187,0.105302
4,kmeans++,10,0.876621,0.329331
5,kcenter,10,0.799212,0.180888


## Problem 5

Suppose you have a population of 1 million people, out of which at least 1% are coffee drinkers. You want to get the estimate of this fraction by using sampling. Give the algorithm and the estimate. What kind of error bounds can you give with probability 99%.

### Solution

First, let's assume that there is a probability $p$ of a person drinking coffee if we choose a random person from the population. Then the average number of people who drink coffee is: $p \cdot 10^6$. Let us first define a series of bernoulli random variables $X_1, X_2, \dots, X_n$ where $X_i = 1$ if the $i^{th}$ person we ask is a coffee drinker and $X_i = 0$ otherwise. Now, we want to estimate $p$ using sampling. The algorithm which we will use to estimate $p$ is as follows:

1. First we will choose $n$ people from the population randomly.
2. We will take a poll of the number of coffee drinkers in the sample.
3. Then our estimate the fraction of the number of coffee drinkers in the population using the following formula:
	$$\frac{10^6}{n} \sum_{i=1}^{n} X_i$$

Now we want to quantify the error which we are getting in our estimate. More formally, we want a relation between $n$ and $\epsilon$ such that we can guarantee that the estimated fraction $\bar{X}$ of people who drink coffee lies in the interval $[p - \epsilon, p + \epsilon]$ with probability atleast 99%. Now we know that from the two sided chernoff bounds:

$$
\begin{align*}
	Pr[|X - \mu| \ge k \mu] &\le 2e^{-\frac{k ^ 2}{2 + k} \mu}\\
\end{align*}
$$
Lets apply this on $X = \sum_{i=1}^{n} X_i$, hence $\mu = np$:

$$
	\begin{align*}
		\implies P\left(\left|\sum_i X_i - n p\right| \ge k n p\right) &\le 2e^{-\frac{k ^ 2}{2 + k} n p}\\
		\implies P\left(\left|\frac{1}{n}\sum_i X_i - p\right| \ge k p\right) &\le 2e^{-\frac{k ^ 2}{2 + k} n p}\\
		\implies P\left(|\bar{X} - p| \ge k p\right) &\le 2e^{-\frac{k ^ 2}{2 + k} n p} \\
		\implies 1 - P\left(|\bar{X} - p| \ge k p\right) &\ge 1 - 2e^{-\frac{k ^ 2}{2 + k} n p} \\
		\implies P\left(|\bar{X} - p| < k p\right) &\ge 1 - 2e^{-\frac{k ^ 2}{2 + k} n p}
	\end{align*}
$$

Since we want the probability to be atleast 99%, we have:

$$
\begin{align*}
	1 - 2e^{-\frac{k ^ 2}{2 + k} n p} &\ge 0.99\\
	\implies e^{-\frac{k ^ 2}{2 + k} n p} &\le 0.005\\
	\implies -\frac{k ^ 2}{2 + k} n p &\le \log(0.005)\\
	\implies \frac{k ^ 2}{2 + k} n p &\ge -\log(0.005)\\
\end{align*}
$$

We know that $k p = \epsilon$, so we can write the above equation as:

$$
\begin{align*}
	\frac{\epsilon ^ 2}{2p + \epsilon} n &\ge -\log(0.005)\\
\end{align*}
$$

Now we know that, $p \le 1$ so using this fact:

$$\frac{\epsilon^2}{2p + \epsilon} n \ge \frac{\epsilon^2}{2 + \epsilon} n$$

Now if $\frac{\epsilon^2}{2 + \epsilon} n \ge -\log(0.005)$, then we are guarenteed to have $\frac{\epsilon^2}{2p + \epsilon} n \ge -\log(0.005)$. So:

$$
\begin{align*}
	\frac{\epsilon^2}{2 + \epsilon} n &\ge -\log(0.005)\\
	\implies n &\ge \frac{-\log(0.005)(2 + \epsilon)}{\epsilon^2}
\end{align*}
$$

Now since atleast 1% of the total people drink coffee, we can safely take $\epsilon = 0.01$. So the number of people we need to sample to get the estimate of the fraction of coffee drinkers with an error of 1% is:

$$
\begin{align*}
	n &\ge \frac{-\log(0.005)(2 + 0.01)}{0.01^2}\\
	\implies n &\ge 106496
\end{align*}
$$

Hence restating our algorithm, if we want our error bound to be $\epsilon$% in estimating the total number of coffee drinkers, then:
1. Sample atleast $n = \frac{-\log(0.005)(2 + \epsilon)}{\epsilon^2}$ people from the population.
2. Take a poll of the number of coffee drinkers in the sample.
3. Estimate the fraction of coffee drinkers in the population using the following formula:
	$$\frac{10^6}{n} \sum_{i=1}^{n} X_i$$

choosing $\epsilon = 1$%, we get $n = 106496$. Hence if we estimate our answer to be: $10^6 \cdot p$, then our answer lies within $\pm 10000$ people of the actual answer. Below we have also verified that our answer is correct.

In [ ]:
# Parameters
total_population = 1000000
fraction_coffee_drinkers = 0.01 # 50% of the population drinks coffee
sample_size = 106496
iterations = 100

population = np.random.choice([0, 1], size=total_population, p=[1 - fraction_coffee_drinkers, fraction_coffee_drinkers])

estimated_fractions = []
within_error_margin_count = 0

for _ in range(iterations):
	sample = np.random.choice(population, size=sample_size, replace=False)
	
	estimated_fraction = np.mean(sample)
	estimated_fractions.append(estimated_fraction)
	
	true_fraction = fraction_coffee_drinkers
	error_margin = 0.01
	if abs(estimated_fraction - true_fraction) <= error_margin:
		within_error_margin_count += 1

percentage_within_error_margin = (within_error_margin_count / iterations) * 100

print(f"Percentage of iterations within 1% error margin: {percentage_within_error_margin}%")

Percentage of iterations within 1% error margin: 98.0%
